# Case 9 — Rolling-threshold EMA freezing (adaptation dependency)

**Reproduces:** Fig 4.17, 4.18

Contextual anomalies, window mode. ema=0 freezes the adaptive decision threshold at its warmup-calibrated value — no online recalibration as the stream evolves. The thesis found MLP-VAE-Cyclic stays roughly as good as its adaptive baseline (its score distribution barely drifts), while Transformer-VAE collapses — its scores keep moving away from the frozen warmup calibration.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the small synthetic ERA5-shaped dataset shipped with the repo (`scripts/generate_mini_era5.py`) — no data download, no license issues. Numbers will differ from the thesis's real-ERA5 figures (much smaller warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# TODO: update this URL once the repo is pushed to GitHub
REPO_URL = "https://github.com/<your-username>/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Generates data/era5/*.csv — fully synthetic, ERA5-shaped, no download needed
!python scripts/generate_mini_era5.py

## Run the suite

`notebooks/cases/case09_ema_freezing_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially on the one Colab GPU; each is small (mini dataset), so the whole case should finish in a few minutes.

In [ ]:
!python run_regression.py notebooks/cases/case09_ema_freezing_suite.yaml \
    --session runs/regression/case09_ema_freezing --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case09_ema_freezing

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case09_ema_freezing/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Display the key comparison plot(s) inline
import glob
from IPython.display import Image, display

print("F1 vs EMA alpha (0 = frozen), split by architecture:")
for p in sorted(glob.glob("runs/regression/case09_ema_freezing/cross_compare/contextual/section_lines_ema.png")):
    display(Image(filename=p))